# Notebook 1: Data Splitting and Model Preparation

This notebook creates consistent training, validation and testing datasets for the baseline and enriched models. The split follows time order so that later observations are not used to predict earlier observations.

The same taxi zones remain in every partition because the research objectives require prediction performance for all analysis-ready zones. `LocationID` is retained only as evaluation metadata and is not treated as a predictor.


## 1. Environment and input files

The two complete model-ready datasets are loaded from the supplied `model_datasets` directory. The six split datasets produced by this notebook are saved back to the same directory for use by the later modelling notebooks.


In [19]:
from pathlib import Path
import os

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

WINDOWS_OUTPUTS = Path(
    r"C:\Asia Pacific University\All Module Notes\Semister-5\Investigations\FYP Semester 1\Progress\fyp\processed_outputs"
)
PROCESSED_OUTPUTS = Path(os.environ.get("NYC_TAXI_PROCESSED_OUTPUTS", WINDOWS_OUTPUTS))
DATA_DIR = PROCESSED_OUTPUTS / "model_datasets"

BASELINE_PATH = DATA_DIR / "nyc_yellow_taxi_baseline_model_dataset_2025_q1.csv"
ENRICHED_PATH = DATA_DIR / "nyc_yellow_taxi_enriched_model_dataset_2025_q1.csv"

print(f"Model dataset directory: {DATA_DIR}")


Model dataset directory: C:\Asia Pacific University\All Module Notes\Semister-5\Investigations\FYP Semester 1\Progress\fyp\processed_outputs\model_datasets


## 2. Load and verify the complete datasets

Only essential checks are repeated here: matching rows, matching targets, no missing modelling values and no duplicate zone-hour observations. The detailed exploration remains in the Data Understanding section.


In [20]:
# Confirm that the complete baseline and enriched datasets are available.
missing_files = [
    str(path) for path in [BASELINE_PATH, ENRICHED_PATH]
    if not path.exists()
]
if missing_files:
    raise FileNotFoundError("Missing model-ready dataset(s):\n" + "\n".join(missing_files))

# Load and sort both datasets using the shared zone-hour keys.
baseline = pd.read_csv(BASELINE_PATH, parse_dates=["pickup_hour"])
enriched = pd.read_csv(ENRICHED_PATH, parse_dates=["pickup_hour"])

KEY_COLUMNS = ["LocationID", "pickup_hour"]
TARGET = "pickup_count"
BASELINE_FEATURES = [
    "hour",
    "day_of_week",
    "month",
    "temperature_f",
    "precipitation_in",
]
CONTEXTUAL_FEATURES = [
    "subway_proximity_km",
    "zero_vehicle_household_rate",
]
ENRICHED_FEATURES = BASELINE_FEATURES + CONTEXTUAL_FEATURES
METADATA_COLUMNS = ["LocationID", "borough", "zone", "pickup_hour"]

baseline = baseline.sort_values(KEY_COLUMNS).reset_index(drop=True)
enriched = enriched.sort_values(KEY_COLUMNS).reset_index(drop=True)

assert len(baseline) == len(enriched)
assert not baseline.duplicated(KEY_COLUMNS).any()
assert not enriched.duplicated(KEY_COLUMNS).any()
assert baseline[KEY_COLUMNS].equals(enriched[KEY_COLUMNS])
assert baseline[TARGET].equals(enriched[TARGET])
assert baseline[METADATA_COLUMNS + [TARGET] + BASELINE_FEATURES].isna().sum().sum() == 0
assert enriched[METADATA_COLUMNS + [TARGET] + ENRICHED_FEATURES].isna().sum().sum() == 0

dataset_check = pd.DataFrame({
    "Check": ["Rows", "Taxi zones", "Unique hours", "Duplicate zone-hours", "Missing modelling values", "Matching keys and targets"],
    "Result": [len(baseline), baseline["LocationID"].nunique(), baseline["pickup_hour"].nunique(), 0, 0, True],
})
display(dataset_check)


,Check,Result
0,Rows,559181
1,Taxi zones,259
2,Unique hours,2159
3,Duplicate zone-hours,0
4,Missing modelling values,0
5,Matching keys and targets,True


## 3. Create the chronological 60/20/20 partition

Unique hourly timestamps are divided in their original order. Complete timestamps are assigned to one partition only, preventing observations from the same hour from being divided between development and testing data.


In [ ]:
# Divide the ordered unique hours into approximately 60%, 20% and 20%.
unique_hours = np.array(sorted(baseline["pickup_hour"].unique()))
n_hours = len(unique_hours)
n_train_hours = int(np.floor(n_hours * 0.60))
n_validation_hours = (n_hours - n_train_hours) // 2

train_hours = unique_hours[:n_train_hours]
validation_hours = unique_hours[n_train_hours:n_train_hours + n_validation_hours]
test_hours = unique_hours[n_train_hours + n_validation_hours:]

hour_partitions = {
    "train": set(train_hours),
    "validation": set(validation_hours),
    "test": set(test_hours),
}

# Confirm that no timestamp appears in more than one partition.
print(f"Training hours: {len(train_hours):,}")
print(f"Validation hours: {len(validation_hours):,}")
print(f"Testing hours: {len(test_hours):,}")


Training hours: 1,295
Validation hours: 432
Testing hours: 432


In [22]:
# Apply the identical timestamp partition to baseline and enriched data.
def create_partitions(frame):
    return {
        "train": frame[frame["pickup_hour"].isin(train_hours)].copy(),
        "validation": frame[frame["pickup_hour"].isin(validation_hours)].copy(),
        "test": frame[frame["pickup_hour"].isin(test_hours)].copy(),
    }


baseline_partitions = create_partitions(baseline)
enriched_partitions = create_partitions(enriched)

# Confirm that every source row was assigned once.
assert sum(len(frame) for frame in baseline_partitions.values()) == len(baseline)
assert sum(len(frame) for frame in enriched_partitions.values()) == len(enriched)

partition_check = pd.DataFrame([
    {
        "Partition": name.title(),
        "Hours": frame["pickup_hour"].nunique(),
        "Rows": len(frame),
        "Share of rows (%)": len(frame) / len(baseline) * 100,
        "Start": frame["pickup_hour"].min(),
        "End": frame["pickup_hour"].max(),
    }
    for name, frame in baseline_partitions.items()
])
display(partition_check.round({"Share of rows (%)": 2}))


,Partition,Hours,Rows,Share of rows (%),Start,End
0,Train,1295,335405,59.98,2025-01-01 00:00:00,2025-02-23 22:00:00
1,Validation,432,111888,20.01,2025-02-23 23:00:00,2025-03-13 23:00:00
2,Test,432,111888,20.01,2025-03-14 00:00:00,2025-03-31 23:00:00


## 4. Confirm the repeated LocationID structure after splitting

The following check supports the earlier Data Understanding finding. Every analysis-ready taxi zone is present in the training, validation and testing periods. Therefore, the identifier repeats across time and must remain evaluation metadata rather than becoming a numerical predictor.


In [23]:
# Compare the unique LocationID sets across all three partitions.
zone_sets = {
    name: set(frame["LocationID"].unique())
    for name, frame in baseline_partitions.items()
}

all_zone_sets_match = (
    zone_sets["train"] == zone_sets["validation"] == zone_sets["test"]
)

location_check = pd.DataFrame([
    {
        "Partition": name.title(),
        "Unique LocationID values": len(zone_ids),
        "Also present in training": len(zone_ids & zone_sets["train"]),
        "Missing from training": len(zone_ids - zone_sets["train"]),
    }
    for name, zone_ids in zone_sets.items()
])

display(location_check)
print("Identical LocationID sets in all partitions:", all_zone_sets_match)
assert all_zone_sets_match


,Partition,Unique LocationID values,Also present in training,Missing from training
0,Train,259,259,0
1,Validation,259,259,0
2,Test,259,259,0


Identical LocationID sets in all partitions: True


In [24]:
# Show how often each LocationID repeats within each partition.
repeat_check = pd.DataFrame([
    {
        "Partition": name.title(),
        "Minimum rows per LocationID": int(frame.groupby("LocationID").size().min()),
        "Maximum rows per LocationID": int(frame.groupby("LocationID").size().max()),
        "Each LocationID repeats": bool((frame.groupby("LocationID").size() > 1).all()),
    }
    for name, frame in baseline_partitions.items()
])
display(repeat_check)


,Partition,Minimum rows per LocationID,Maximum rows per LocationID,Each LocationID repeats
0,Train,1295,1295,True
1,Validation,432,432,True
2,Test,432,432,True


## 5. Define predictors, target and evaluation metadata

The saved split files retain all columns because later zone-level evaluation needs `LocationID`, borough, zone and pickup time. These metadata columns are excluded when the predictor matrices are constructed in the modelling notebooks.

The same exclusion must be applied to training, validation and testing. Otherwise, the model would receive a different column structure between development and final evaluation.


In [25]:
# Display the consistent modelling roles assigned to every column group.
column_roles = pd.DataFrame({
    "Role": ["Baseline predictors", "Additional enriched predictors", "Target", "Evaluation metadata"],
    "Columns": [
        ", ".join(BASELINE_FEATURES),
        ", ".join(CONTEXTUAL_FEATURES),
        TARGET,
        ", ".join(METADATA_COLUMNS),
    ],
    "Supplied to baseline model": [True, False, False, False],
    "Supplied to enriched model": [True, True, False, False],
})
display(column_roles)


,Role,Columns,Supplied to baseline model,Supplied to enriched model
0,Baseline predictors,"hour, day_of_week, month, temperature_f, preci...",True,True
1,Additional enriched predictors,"subway_proximity_km, zero_vehicle_household_rate",False,True
2,Target,pickup_count,False,False
3,Evaluation metadata,"LocationID, borough, zone, pickup_hour",False,False


In [26]:
# Verify that baseline and enriched keys remain aligned within every partition.
alignment_rows = []
for partition in ["train", "validation", "test"]:
    baseline_frame = baseline_partitions[partition].reset_index(drop=True)
    enriched_frame = enriched_partitions[partition].reset_index(drop=True)
    matching_keys = baseline_frame[KEY_COLUMNS].equals(enriched_frame[KEY_COLUMNS])
    matching_target = baseline_frame[TARGET].equals(enriched_frame[TARGET])
    alignment_rows.append({
        "Partition": partition.title(),
        "Matching rows": len(baseline_frame) == len(enriched_frame),
        "Matching zone-hour keys": matching_keys,
        "Matching target values": matching_target,
    })
    assert matching_keys and matching_target

display(pd.DataFrame(alignment_rows))


,Partition,Matching rows,Matching zone-hour keys,Matching target values
0,Train,True,True,True
1,Validation,True,True,True
2,Test,True,True,True


## 6. Export the prepared split datasets

No figure files or separate split-summary file are produced. The six model datasets are the only outputs required from this notebook.


In [18]:
# Save the six prepared datasets using stable names expected by later notebooks.
output_paths = []
for dataset_name, partitions in [
    ("baseline", baseline_partitions),
    ("enriched", enriched_partitions),
]:
    for partition, frame in partitions.items():
        output_path = DATA_DIR / f"nyc_yellow_taxi_{dataset_name}_{partition}_2025_q1.csv"
        frame.to_csv(output_path, index=False)
        output_paths.append(output_path)

print("Saved model split datasets:")
for output_path in output_paths:
    print(f"- {output_path.name}")

# Reload each saved file and confirm that the export is complete.
export_checks = []
for dataset_name, partitions in [
    ("baseline", baseline_partitions),
    ("enriched", enriched_partitions),
]:
    for partition, expected_frame in partitions.items():
        output_path = DATA_DIR / f"nyc_yellow_taxi_{dataset_name}_{partition}_2025_q1.csv"
        saved = pd.read_csv(output_path, usecols=["LocationID", "pickup_hour"])
        expected_rows = len(expected_frame)
        expected_zones = expected_frame["LocationID"].nunique()
        export_checks.append({
            "Dataset": dataset_name.title(),
            "Partition": partition.title(),
            "Rows saved": len(saved),
            "Expected rows": expected_rows,
            "Taxi zones saved": saved["LocationID"].nunique(),
            "Export verified": len(saved) == expected_rows and saved["LocationID"].nunique() == expected_zones,
        })
        assert len(saved) == expected_rows
        assert saved["LocationID"].nunique() == expected_zones

display(pd.DataFrame(export_checks))


Saved model split datasets:
- nyc_yellow_taxi_baseline_train_2025_q1.csv
- nyc_yellow_taxi_baseline_validation_2025_q1.csv
- nyc_yellow_taxi_baseline_test_2025_q1.csv
- nyc_yellow_taxi_enriched_train_2025_q1.csv
- nyc_yellow_taxi_enriched_validation_2025_q1.csv
- nyc_yellow_taxi_enriched_test_2025_q1.csv


,Dataset,Partition,Rows saved,Expected rows,Taxi zones saved,Export verified
0,Baseline,Train,335405,335405,259,True
1,Baseline,Validation,111888,111888,259,True
2,Baseline,Test,111888,111888,259,True
3,Enriched,Train,335405,335405,259,True
4,Enriched,Validation,111888,111888,259,True
5,Enriched,Test,111888,111888,259,True
